In [1]:
import pandas as pd

df = pd.read_csv("/kaggle/input/datasets/ahmedmohameddawoud/ecommerce-ab-testing/ab_test.csv")



In [2]:
df.columns

Index(['id', 'time', 'con_treat', 'page', 'converted'], dtype='object')

In [3]:
df.head

<bound method NDFrame.head of             id     time  con_treat      page  converted
0       851104  11:48.6    control  old_page          0
1       804228  01:45.2    control  old_page          0
2       661590  55:06.2  treatment  new_page          0
3       853541  28:03.1  treatment  new_page          0
4       864975  52:26.2    control  old_page          1
...        ...      ...        ...       ...        ...
294473  751197  28:38.6    control  old_page          0
294474  945152  51:57.1    control  old_page          0
294475  734608  45:03.4    control  old_page          0
294476  697314  20:29.0    control  old_page          0
294477  715931  40:24.5  treatment  new_page          0

[294478 rows x 5 columns]>

In [4]:
df.shape


(294478, 5)

In [5]:
df["id"].nunique()

290584

In [6]:
df["id"].value_counts()

id
752737    2
781280    2
767913    2
886060    2
731779    2
         ..
653383    1
732573    1
867433    1
899189    1
900200    1
Name: count, Length: 290584, dtype: int64

In [7]:
df["page"].value_counts()

page
old_page    147239
new_page    147239
Name: count, dtype: int64

In [8]:
print("Rows:", len(df))
print("Unique users:", df["id"].nunique())

Rows: 294478
Unique users: 290584


In [9]:
duplicates = df["id"].value_counts()
print("Users appearing more than once:", (duplicates > 1).sum())

Users appearing more than once: 3894


In [10]:
user_pages = df.groupby("id")["page"].nunique()

print("Users who saw multiple pages:", (user_pages > 1).sum())

Users who saw multiple pages: 1998


In [11]:
duplicates = df["id"].value_counts()

duplicates.value_counts().sort_index()

count
1    286690
2      3894
Name: count, dtype: int64

In [12]:
problem_users = user_pages[user_pages > 1].index

df[df["id"].isin(problem_users)].sort_values("id").head(20)

,id,time,con_treat,page,converted
213114,630052,25:54.1,treatment,old_page,1
230259,630052,16:05.2,treatment,new_page,0
22513,630126,35:54.8,treatment,old_page,0
251762,630126,16:00.3,treatment,new_page,0
11792,630137,59:22.1,control,new_page,0
183371,630137,08:49.9,control,old_page,0
96929,630471,14:17.4,control,new_page,0
110634,630471,42:51.5,control,old_page,0
1282,630780,14:27.7,control,old_page,0
201303,630780,27:53.9,control,new_page,0


In [13]:
pd.crosstab(df["con_treat"], df["page"])

page,new_page,old_page
con_treat,,
control,1928,145274
treatment,145311,1965


In [14]:
clean_df = df[
    ((df["con_treat"] == "control") & (df["page"] == "old_page")) |
    ((df["con_treat"] == "treatment") & (df["page"] == "new_page"))
]

In [15]:
print("Original rows:", len(df))
print("Clean rows:", len(clean_df))
print("Rows removed:", len(df) - len(clean_df))

Original rows: 294478
Clean rows: 290585
Rows removed: 3893


In [16]:
pd.crosstab(clean_df["con_treat"], clean_df["page"])

page,new_page,old_page
con_treat,,
control,0,145274
treatment,145311,0


In [17]:
from scipy.stats import chisquare

observed = [
    clean_df[clean_df["con_treat"] == "control"].shape[0],
    clean_df[clean_df["con_treat"] == "treatment"].shape[0]
]

expected = [sum(observed) / 2, sum(observed) / 2]

chisquare(observed, f_exp=expected)

Power_divergenceResult(statistic=np.float64(0.0047111860557151955), pvalue=np.float64(0.9452777066998205))

In [18]:
control = clean_df[clean_df["con_treat"] == "control"]
treatment = clean_df[clean_df["con_treat"] == "treatment"]

control_conversion = control["converted"].mean()
treatment_conversion = treatment["converted"].mean()

print("Control conversion rate:", control_conversion)
print("Treatment conversion rate:", treatment_conversion)

Control conversion rate: 0.1203863045004612
Treatment conversion rate: 0.11880724790277405


In [19]:
absolute_difference = treatment_conversion - control_conversion

relative_uplift = (
    (treatment_conversion - control_conversion)
    / control_conversion
) * 100

print("Absolute difference:", absolute_difference)
print("Relative uplift:", relative_uplift, "%")

Absolute difference: -0.0015790565976871451
Relative uplift: -1.31165800315857 %


In [20]:
control = clean_df[clean_df["con_treat"] == "control"]
treatment = clean_df[clean_df["con_treat"] == "treatment"]

In [21]:
from statsmodels.stats.proportion import proportions_ztest

control_converted = control["converted"].sum()
treatment_converted = treatment["converted"].sum()

successes = [control_converted, treatment_converted]
samples = [len(control), len(treatment)]

z_stat, p_value = proportions_ztest(successes, samples)

print("Z-statistic:", z_stat)
print("P-value:", p_value)

Z-statistic: 1.3116075339133115
P-value: 0.18965258971881804


In [22]:
from statsmodels.stats.power import NormalIndPower
from statsmodels.stats.proportion import proportion_effectsize

baseline = 0.1204
target = 0.1404

effect_size = proportion_effectsize(baseline, target)

analysis = NormalIndPower()

sample_size = analysis.solve_power(
    effect_size=effect_size,
    alpha=0.05,
    power=0.80,
    alternative="two-sided"
)

print("Required sample size per group:", round(sample_size))
print("Required total sample size:", round(sample_size * 2))

Required sample size per group: 4444
Required total sample size: 8888


In [23]:
import numpy as np
from statsmodels.stats.proportion import confint_proportions_2indep

control_converted = control["converted"].sum()
treatment_converted = treatment["converted"].sum()

control_n = len(control)
treatment_n = len(treatment)

lower, upper = confint_proportions_2indep(
    treatment_converted,
    treatment_n,
    control_converted,
    control_n,
    method="wald"
)

difference = treatment["converted"].mean() - control["converted"].mean()

print("Treatment - Control difference:", difference)
print("95% CI lower:", lower)
print("95% CI upper:", upper)

Treatment - Control difference: -0.0015790565976871451
95% CI lower: -0.00393867032963623
95% CI upper: 0.0007805571342619391
